# Capstone Project — Track C: Multi-Source Knowledge Base with Routing
**Team members:** _[FILL IN: your full names — required, submissions without names are not graded]_

**Course:** Building Agentic AI Systems

**What this notebook does:** UniGuide AI answers student questions by routing between **two separate
knowledge sources** — Academic Regulations and Campus Services — using an LLM-driven router (not
keyword matching), retrieves from the correct source with real embeddings, pauses for human approval
on sensitive/uncertain answers, and remembers both the current conversation and durable facts about
the student across sessions.

Every `# TODO` below is something you need to fill in. The surrounding structure already works —
your job is to complete the blanks so the whole thing runs end-to-end. Run cells top to bottom in
Google Colab.

You need a **free Groq API key** (https://console.groq.com) and, optionally but required for full
credit, a **free LangSmith API key** (https://smith.langchain.com) — both go in Colab Secrets
(the key icon on the left sidebar), never hardcoded in the notebook.

## 0. Install dependencies

In [1]:
%pip install -q -U \
    langgraph \
    langchain \
    langchain-groq \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    sentence-transformers \
    langsmith \
    pydantic

## 1. API keys and LangSmith tracing

**Important:** the correct environment variable for LangGraph/LangChain tracing is
`LANGCHAIN_TRACING_V2` (some notebooks online use `LANGSMITH_TRACING_V2`, which is **not**
a real variable and silently does nothing — double check you have the name below exactly).

In [2]:
import os
from google.colab import userdata

# --- Groq (required) ---
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# LangSmith (optional, for observability)
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
# os.environ["LANGCHAIN_PROJECT"] = "uniguide-capstone"

print("Keys loaded.")

Keys loaded.


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # free tier, tool-calling capable
    temperature=0,
)

print(llm.invoke("Reply with exactly: UniGuide AI is ready.").content)

UniGuide AI is ready.


## 2. Two distinct knowledge sources

This is the part of the original submission that needs the real fix: there were two "sources" in
name only — one shared vector store was searched no matter what the router said. Below, fill in
**two separate document lists**, and further down we build **two separate vector stores**, one per
source, so routing actually changes what gets searched.

In [4]:
from langchain_core.documents import Document

# TODO: replace/expand with your own real content (at least 2-3 short passages per source).
# Keep the metadata={"source": "..."} — the router and retriever tools key off this.

academic_docs = [
    Document(
        page_content="The university's attendance policy requires students to attend at least 80% of all scheduled classes. Absences exceeding this limit may result in a failing grade for the course. Medical emergencies and approved university activities are exceptions, provided proper documentation is submitted to the course instructor within one week of the absence.",
        metadata={"source": "academic_regulations"},
    ),
    Document(
        page_content="Grades are assigned on a standard scale: A (90-100%), B (80-89%), C (70-79%), D (60-69%), and F (below 60%). Pass/Fail options are available for certain elective courses. Students can appeal a final grade within 30 days of its posting by submitting a formal appeal to the Registrar's Office.",
        metadata={"source": "academic_regulations"},
    ),
    Document(
        page_content="Withdrawal from a course is permitted until the 10th week of the semester without academic penalty. After the 10th week, a 'W' (Withdrawal) grade will be recorded, which does not affect GPA but remains on the transcript. Complete withdrawal from all courses requires consultation with an academic advisor.",
        metadata={"source": "academic_regulations"},
    ),
]

campus_docs = [
    Document(
        page_content="The main library operates from 8 AM to 10 PM on weekdays and 10 AM to 6 PM on weekends. Extended hours are available during exam periods. Resources include quiet study zones, group project rooms, and access to a vast digital collection of journals and e-books. Laptops can be borrowed for up to 4 hours.",
        metadata={"source": "campus_services"},
    ),
    Document(
        page_content="The IT Helpdesk provides support for campus Wi-Fi, student email, software installations, and account password resets. They are located on the first floor of the Student Union and are open from 9 AM to 5 PM, Monday through Friday. Online support is also available via the university's portal.",
        metadata={"source": "campus_services"},
    ),
    Document(
        page_content="Student wellbeing services offer free counseling, mental health resources, and workshops on stress management and resilience. Appointments can be scheduled confidentially through the health services portal or by visiting their office on the third floor of the Health & Wellness Center. Emergency support is available 24/7.",
        metadata={"source": "campus_services"},
    ),
]

## 3. Split, embed, and store — as two SEPARATE vector stores

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

academic_chunks = splitter.split_documents(academic_docs)
campus_chunks = splitter.split_documents(campus_docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Two separate stores — this is what makes routing actually mean something.
academic_store = Chroma.from_documents(academic_chunks, embeddings, collection_name="academic")
campus_store = Chroma.from_documents(campus_chunks, embeddings, collection_name="campus")

academic_retriever = academic_store.as_retriever(search_kwargs={"k": 2})
campus_retriever = campus_store.as_retriever(search_kwargs={"k": 2})

print("Two independent vector stores are ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Two independent vector stores are ready.


## 4. Retriever tools

Real tools that query the vector stores at runtime — not hardcoded strings.

In [6]:
from langchain_core.tools import tool

@tool
def search_academic_regulations(query: str) -> str:
    """Search academic regulations: grading, attendance, exams, withdrawal, academic policy."""
    docs = academic_retriever.invoke(query)
    if not docs:
        return "No relevant academic-regulations information found."
    return "\n\n".join(d.page_content for d in docs)

@tool
def search_campus_services(query: str) -> str:
    """Search campus services: library, IT support, careers, student wellbeing."""
    docs = campus_retriever.invoke(query)
    if not docs:
        return "No relevant campus-services information found."
    return "\n\n".join(d.page_content for d in docs)

print("Retriever tools ready.")

Retriever tools ready.


## 5. LLM-driven routing (structured output, not keyword matching)

TODO: this is the second real fix — the original router used `if "grade" in question: ...`
keyword matching. Here the **LLM itself** classifies the question using a Pydantic schema, so the
model is making the routing decision, not a string search.

In [7]:
from typing import Literal
from pydantic import BaseModel, Field

class RouteDecision(BaseModel):
    destination: Literal["academic_regulations", "campus_services", "both"] = Field(
        description="academic_regulations for questions about policies like grading, attendance, or course withdrawal; campus_services for questions about university facilities like the library or IT support, or student wellbeing; both if the question requires information from both academic and campus sources."
    )
    reason: str = Field(description="One short sentence explaining the routing choice.")

router_llm = llm.with_structured_output(RouteDecision)

def classify_question(question: str) -> RouteDecision:
    # TODO: you can adjust this prompt if the router misclassifies your test questions.
    return router_llm.invoke(f"Classify this student question for routing:\n\n{question}")

# Quick sanity check before wiring it into the graph:
test_route = classify_question("What is the policy for withdrawing from a course?")
print(test_route)

destination='academic_regulations' reason='The question is about course withdrawal policy, which is an academic regulation.'


## 6. Short-term and long-term memory

- **Short-term:** `InMemorySaver` — checkpoints this conversation thread so a paused `interrupt()`
  can resume exactly where it left off.
- **Long-term:** `InMemoryStore` — TODO: pick one durable fact about the student that should
  persist *across different threads/sessions* (not just this conversation). Examples: their major,
  a preference for concise vs. detailed answers, or a running count of questions asked.

In [8]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

checkpointer = InMemorySaver()
long_term_store = InMemoryStore()

def remember_student_fact(student_id: str, key: str, value):
    # TODO: namespace + key can stay as-is, just make sure you actually call this somewhere
    # in the workflow below (not just define it and never use it).
    long_term_store.put(("students", student_id), key, {"value": value})

def recall_student_fact(student_id: str, key: str):
    item = long_term_store.get(("students", student_id), key)
    return item.value["value"] if item else None

## 7. Functional API workflow — `@task` / `@entrypoint`

TODO: this replaces the original `StateGraph`. Error handling required: a `RetryPolicy` for
transient failures (already wired below) plus the `interrupt()` for human review (also below) —
that's your 2 of the 4 required error-handling strategies.

In [9]:
from langgraph.func import entrypoint, task
from langgraph.types import interrupt, Command, RetryPolicy

@task(retry_policy=RetryPolicy(max_attempts=3))
def route_task(question: str) -> RouteDecision:
    return classify_question(question)

@task(retry_policy=RetryPolicy(max_attempts=3))
def retrieve_task(question: str, destination: str) -> str:
    if destination == "academic_regulations":
        return search_academic_regulations.invoke(question)
    elif destination == "campus_services":
        return search_campus_services.invoke(question)
    else:  # "both"
        return (
            search_academic_regulations.invoke(question)
            + "\n\n"
            + search_campus_services.invoke(question)
        )

@task(retry_policy=RetryPolicy(max_attempts=3))
def generate_answer_task(question: str, context: str) -> str:
    # TODO: adjust this prompt template if your answers aren't grounded enough.
    prompt = f"""Answer the student's question using ONLY the information below.
If the information doesn't contain the answer, say so honestly.

Context:
{context}

Question:
{question}
"""
    return llm.invoke(prompt).content


@entrypoint(checkpointer=checkpointer, store=long_term_store)
def uniguide_workflow(inputs: dict) -> dict:
    question = inputs["question"]
    student_id = inputs.get("student_id", "anonymous")

    route = route_task(question).result()
    context = retrieve_task(question, route.destination).result()

    # Implement needs_review condition: trigger review if routing to "both"
    needs_review = (route.destination == "both")

    draft = generate_answer_task(question, context).result()

    if needs_review:
        decision = interrupt({
            "question": question,
            "route": route.destination,
            "draft": draft,
            "message": "Review this answer. Reply with 'approve' or a corrected answer.",
        })
        final_answer = draft if str(decision).strip().lower() == "approve" else str(decision)
    else:
        final_answer = draft

    # Call remember_student_fact to store/update question count
    current_count = recall_student_fact(student_id, "question_count")
    if current_count is None:
        current_count = 0
    remember_student_fact(student_id, "question_count", current_count + 1)

    return {
        "question": question,
        "route": route.destination,
        "reason": route.reason,
        "answer": final_answer,
        "student_question_count": current_count + 1,
    }

print("Workflow compiled.")

Workflow compiled.


## 8. Demo tests

TODO: replace each question below with your own, and add a screenshot/printed output for each once it runs — this is what the grader checks against your write-up.

In [10]:
import uuid

config_a = {"configurable": {"thread_id": str(uuid.uuid4())}}
result_a = uniguide_workflow.invoke(
    {"question": "What is the university's attendance policy?", "student_id": "student-1"},
    config=config_a,
)
print(result_a)

{'question': "What is the university's attendance policy?", 'route': 'academic_regulations', 'reason': 'The question is about attendance policy, which is an academic regulation.', 'answer': "The university's attendance policy requires students to attend at least 80% of all scheduled classes. Absences exceeding this limit may result in a failing grade for the course, unless they are due to medical emergencies or approved university activities with proper documentation.", 'student_question_count': 1}


In [11]:
config_b = {"configurable": {"thread_id": str(uuid.uuid4())}}
result_b = uniguide_workflow.invoke(
    {"question": "What are the library operating hours?", "student_id": "student-1"},
    config=config_b,
)
print(result_b)

{'question': 'What are the library operating hours?', 'route': 'campus_services', 'reason': 'The question is about university facilities.', 'answer': 'The library operating hours are:\n- 8 AM to 10 PM on weekdays\n- 10 AM to 6 PM on weekends\nNote: Extended hours are available during exam periods, but the specific hours are not specified.', 'student_question_count': 2}


In [12]:
# This test should trigger your `needs_review` condition from section 7.
config_c = {"configurable": {"thread_id": str(uuid.uuid4())}}
result_c = uniguide_workflow.invoke(
    {"question": "I need to appeal a grade, and I'm also looking for IT support. Can you help with both?", "student_id": "student-1"},
    config=config_c,
)
print(result_c)

if "__interrupt__" in result_c:
    print("\nPaused for human review:")
    print(result_c["__interrupt__"][0].value)

{'__interrupt__': [Interrupt(value={'question': "I need to appeal a grade, and I'm also looking for IT support. Can you help with both?", 'route': 'both', 'draft': "For the grade appeal, you can submit a formal appeal to the Registrar's Office within 30 days of the grade's posting.\n\nFor IT support, you can visit the IT Helpdesk on the first floor of the Student Union, which is open from 9 AM to 5 PM, Monday through Friday. Alternatively, you can also access online support via the university's portal. They can assist you with campus Wi-Fi, student email, software installations, and account password resets.", 'message': "Review this answer. Reply with 'approve' or a corrected answer."}, id='076a8dbabcceb4ff236ee0cba23ed277')]}

Paused for human review:
{'question': "I need to appeal a grade, and I'm also looking for IT support. Can you help with both?", 'route': 'both', 'draft': "For the grade appeal, you can submit a formal appeal to the Registrar's Office within 30 days of the grade'

In [13]:
# Resume the paused workflow from the cell above — only run this if the previous cell paused.
from langgraph.types import Command

resumed = uniguide_workflow.invoke(Command(resume="approve"), config=config_c)
print(resumed)

{'question': "I need to appeal a grade, and I'm also looking for IT support. Can you help with both?", 'route': 'both', 'reason': 'The question requires information from both academic and campus sources.', 'answer': "For the grade appeal, you can submit a formal appeal to the Registrar's Office within 30 days of the grade's posting.\n\nFor IT support, you can visit the IT Helpdesk on the first floor of the Student Union, which is open from 9 AM to 5 PM, Monday through Friday. Alternatively, you can also access online support via the university's portal. They can assist you with campus Wi-Fi, student email, software installations, and account password resets.", 'student_question_count': 3}


In [14]:
# A "both" question that needs information from academic_regulations AND campus_services.
config_d = {"configurable": {"thread_id": str(uuid.uuid4())}}
result_d = uniguide_workflow.invoke(
    {"question": "How can I withdraw from a course and what are the student wellbeing services available?", "student_id": "student-1"},
    config=config_d,
)
print(result_d)

{'__interrupt__': [Interrupt(value={'question': 'How can I withdraw from a course and what are the student wellbeing services available?', 'route': 'both', 'draft': "To withdraw from a course, you are permitted to do so until the 10th week of the semester without academic penalty. After the 10th week, a 'W' (Withdrawal) grade will be recorded, which does not affect GPA but remains on the transcript. Complete withdrawal from all courses requires consultation with an academic advisor.\n\nAs for student wellbeing services, they offer free counseling, mental health resources, and workshops on stress management and resilience. You can schedule appointments confidentially through the health services portal or by visiting their office on the third floor of the Health & Wellness Center.", 'message': "Review this answer. Reply with 'approve' or a corrected answer."}, id='58b23b5f6999ce72e7c71c17870f84fb')]}


## 9. LangSmith

Run this after the demos above, then open your project in LangSmith and confirm you can see the traces before writing the observability paragraph below.

In [15]:
from langchain_core.tracers.langchain import wait_for_all_tracers

wait_for_all_tracers()
print("Traces flushed. Open smith.langchain.com and check project:", os.environ.get("LANGCHAIN_PROJECT"))

Traces flushed. Open smith.langchain.com and check project: None


## 10. Write-up — one short paragraph per section

TODO: fill in every section below with what your notebook *actually does*, based on the demo
outputs above — this is what separates "followed the template" from "understood it," and it's
how grading verifies understanding. Don't leave the TODOs in.

**Agent fundamentals.** TODO — describe the real tool calls (`search_academic_regulations`,
`search_campus_services`) and where structured output is used.

**Multi-agent / routing architecture (Track C).** TODO — describe how `route_task` decides between
your two sources, and point to a demo cell where you can see it choosing correctly.

**RAG pipeline.** TODO — describe your load → split → embed → store → retrieve pipeline, and state
which RAG style you used (2-Step / Agentic / Hybrid) and *why*.

**Context & state management.** TODO — describe your short-term (`InMemorySaver`) vs. long-term
(`InMemoryStore`) split, and point to the specific fact your `remember_student_fact` call stores.

**Human-in-the-loop.** TODO — describe your `needs_review` condition and point to the demo cell
where the interrupt actually paused and then resumed.

**LangGraph functional API & error handling.** TODO — name your two error-handling strategies
(hint: `RetryPolicy` = transient retry, the review pause = user-fixable interrupt) and where each
is in the code.

**Workflow pattern.** TODO — name the pattern you implemented (Routing) and why it fits this
problem.

**LangSmith observability.** TODO — after opening a real trace in your project, write what it
actually showed you (a bottleneck, a routing decision, a retry firing) — not what you expect it to
show.